In [ ]:

import os
import time
import random
import warnings
import numpy as np

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
warnings.filterwarnings("ignore", message=".*unauthenticated.*", category=UserWarning)
warnings.filterwarnings("ignore", message=".*HF Hub.*", category=UserWarning)

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score,
    cohen_kappa_score, matthews_corrcoef,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from PIL import Image

try:
    from thop import profile, clever_format
    THOP_OK = True
except Exception:
    THOP_OK = False

try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_OK = True
except Exception:
    IMBLEARN_OK = False

# =============================================================================
# CONFIG
# =============================================================================
RANDOM_SEED         = 42
IMG_SIZE            = 224
BATCH_SIZE          = 32
EPOCHS_CNN          = 100
EPOCHS_VIT          = 100
EARLY_STOP_PATIENCE = 10
LR                  = 1e-4
WEIGHT_DECAY        = 1e-2

TEST_SPLIT          = 0.15     # locked test
VAL_SPLIT_OF_REMAIN = 0.1765   # ~15% overall val, ~70% train

USE_SMOTE           = True
SMOTE_K             = 5

XGB_PARAMS = dict(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 1,
    reg_alpha        = 0.05,
    reg_lambda       = 0.5,
    objective        = "multi:softprob",
    tree_method      = "hist",
    eval_metric      = "mlogloss",
    n_jobs           = -1,
)

GRADCAM_IMAGE_PATHS = [
    "/kaggle/input/datasets/hadeelsharaf/full-diseases-ds/Diseases of date palm leaves dataset/Infected Date Palm Leaves Dataset/Processed/2. Manganese Deficiency/M7  (100).png",
    "/kaggle/input/datasets/hadeelsharaf/full-diseases-ds/Diseases of date palm leaves dataset/Infected Date Palm Leaves Dataset/Processed/4. Black Scorch/M2_ (102).png",
    "/kaggle/input/datasets/hadeelsharaf/full-diseases-ds/Diseases of date palm leaves dataset/Infected Date Palm Leaves Dataset/Processed/7. Rachis Blight/M5 (10).png",
]

def _get_paths():
    if os.path.exists("/kaggle"):
        data_dir = os.environ.get(
            "INPUT_PATH",
            "/kaggle/input/datasets/hadeelsharaf/full-diseases-ds/Diseases of date palm leaves dataset/Infected Date Palm Leaves Dataset/Processed",
        )
        return data_dir, "/kaggle/working"
    return (r"c:\Users\Azur Computer\Desktop\Processed",
            r"c:\Users\Azur Computer\Desktop\Processed")

BASE_DATA_DIR, OUTPUT_DIR = _get_paths()
DATA_DIR = os.path.join(BASE_DATA_DIR, "dataset")
if not os.path.exists(DATA_DIR):
    DATA_DIR = BASE_DATA_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

# =============================================================================
# DETERMINISM
# =============================================================================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(RANDOM_SEED)

# =============================================================================
# DATA
# =============================================================================
def rgb_loader(path):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return Image.open(path).convert("RGB")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full_ds     = datasets.ImageFolder(DATA_DIR, loader=rgb_loader)
targets     = np.array([s[1] for s in full_ds.samples])
class_names = full_ds.classes
NUM_CLASSES = len(class_names)

# ---- THREE-WAY STRATIFIED SPLIT: train / val / test ----
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SPLIT, random_state=RANDOM_SEED)
trainval_idx, test_idx = next(sss_test.split(np.zeros(len(targets)), targets))
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT_OF_REMAIN, random_state=RANDOM_SEED)
tr_rel, vl_rel = next(sss_val.split(np.zeros(len(trainval_idx)), targets[trainval_idx]))
train_idx = trainval_idx[tr_rel]
val_idx   = trainval_idx[vl_rel]
print(f"  Split -> train={len(train_idx)} | val={len(val_idx)} | test={len(test_idx)}")

def make_loader(idxs, tfms, shuffle):
    ds = Subset(datasets.ImageFolder(full_ds.root, transform=tfms, loader=rgb_loader), list(idxs))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=(device == "cuda"),
                      worker_init_fn=lambda wid: seed_everything(RANDOM_SEED + wid))

train_loader    = make_loader(train_idx, train_tfms, True)
train_loader_na = make_loader(train_idx, eval_tfms,  False)   # no-aug for stable probs
val_loader      = make_loader(val_idx,   eval_tfms,  False)
test_loader     = make_loader(test_idx,  eval_tfms,  False)

# =============================================================================
# TRAIN / PREDICT
# =============================================================================
@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        L.append(model(x).cpu().numpy()); Y.append(y.numpy())
    return np.vstack(L), np.concatenate(Y)

@torch.no_grad()
def predict_softmax(model, loader):
    logits, y = predict_logits(model, loader)
    return torch.softmax(torch.from_numpy(logits), dim=1).numpy(), y

def plot_training_curves(history, name):
    ep = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(ep, history["train_loss"], "b-", label="Train Loss")
    ax[0].plot(ep, history["val_loss"], "r--", label="Val Loss")
    ax[0].set_title(f"{name} - Loss per Epoch"); ax[0].set_xlabel("Epochs"); ax[0].set_ylabel("Loss")
    ax[0].legend(); ax[0].grid(True)
    ax[1].plot(ep, history["train_acc"], "b-", label="Train Accuracy")
    ax[1].plot(ep, history["val_acc"], "g--", label="Val Accuracy")
    ax[1].set_title(f"{name} - Accuracy per Epoch"); ax[1].set_xlabel("Epochs"); ax[1].set_ylabel("Accuracy")
    ax[1].legend(); ax[1].grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{name}_training_curves.png"), dpi=200)
    plt.close()
    print(f"  Saved training curves to {name}_training_curves.png")

def train_backbone(model_name, epochs, tag):
    seed_everything(RANDOM_SEED)
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    amp_dev = "cuda" if device == "cuda" else "cpu"
    scaler  = torch.amp.GradScaler(amp_dev, enabled=(device == "cuda"))

    best_val, best_state, no_improve = -1.0, None, 0
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for ep in range(1, epochs + 1):
        model.train(); running, tp, tl = 0.0, [], []
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(amp_dev, enabled=(device == "cuda")):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
            running += loss.item() * x.size(0)
            tp.append(out.detach().argmax(1).cpu()); tl.append(y.cpu())
        tr_loss = running / len(train_loader.dataset)
        tr_acc  = accuracy_score(torch.cat(tl).numpy(), torch.cat(tp).numpy())
        logits, y_ep = predict_logits(model, val_loader)
        v_acc = accuracy_score(y_ep, logits.argmax(1))
        v_loss = criterion(torch.from_numpy(logits).float().to(device),
                           torch.from_numpy(y_ep).long().to(device)).item()
        hist["train_loss"].append(tr_loss); hist["val_loss"].append(v_loss)
        hist["train_acc"].append(tr_acc);   hist["val_acc"].append(v_acc)
        print(f"  [{tag}] Ep {ep:03d}/{epochs} | tr_loss={tr_loss:.4f} | tr_acc={tr_acc:.4f} | "
              f"val_loss={v_loss:.4f} | val_acc={v_acc:.4f}")
        if v_acc > best_val:
            best_val, no_improve = v_acc, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f"  [{tag}] Early stopping at epoch {ep} (best val_acc={best_val:.4f})")
            break

    plot_training_curves(hist, tag)
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# =============================================================================
# FUSION / SMOTE / XGB / METRICS
# =============================================================================
def smote_balance(X, y):
    if not (USE_SMOTE and IMBLEARN_OK):
        return X, y
    cnt = np.bincount(y, minlength=NUM_CLASSES)
    if cnt.min() <= 1:
        return X, y
    k = max(1, min(SMOTE_K, int(cnt.min()) - 1))
    return SMOTE(random_state=RANDOM_SEED, k_neighbors=k).fit_resample(X, y)

def stacked_features(P_rep, P_pool, alpha, beta):
    return np.concatenate([alpha * P_rep, beta * P_pool], axis=1)

def grid_search_weights(P_rep_val, P_pool_val, y_val):
    best_a, best_s = 0.5, -1.0
    for a in np.linspace(0, 1, 101):
        b = 1 - a
        s = f1_score(y_val, (a * P_rep_val + b * P_pool_val).argmax(1),
                     average="macro", zero_division=0)
        if s > best_s:
            best_s, best_a = s, a
    return best_a, 1 - best_a

def train_xgb(X_tr, y_tr, X_eval, y_eval):
    weights = compute_sample_weight(class_weight="balanced", y=y_tr)
    clf = XGBClassifier(**XGB_PARAMS, num_class=NUM_CLASSES,
                        random_state=RANDOM_SEED, early_stopping_rounds=30)
    clf.fit(X_tr, y_tr, sample_weight=weights,
            eval_set=[(X_tr, y_tr), (X_eval, y_eval)], verbose=False)
    return clf

def evaluate(y_true, y_pred, y_proba):
    if len(y_true) < 2:
        return dict(acc=0, f1_macro=0, precision=0, recall=0, kappa=0, mcc=0, roc_mac=0)
    y_bin = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    try:
        roc_mac = roc_auc_score(y_bin, y_proba, average="macro", multi_class="ovr")
    except Exception:
        roc_mac = 0.0
    return dict(
        acc=accuracy_score(y_true, y_pred),
        f1_macro=f1_score(y_true, y_pred, average="macro", zero_division=0),
        precision=precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall=recall_score(y_true, y_pred, average="macro", zero_division=0),
        kappa=cohen_kappa_score(y_true, y_pred),
        mcc=matthews_corrcoef(y_true, y_pred),
        roc_mac=roc_mac,
    )

# =============================================================================
# GRAD-CAM
# =============================================================================
def get_target_layer(model, name):
    if "RepVGG" in name and hasattr(model, "stage4"):
        return model.stage4[-1]
    if "PoolFormer" in name and hasattr(model, "network"):
        last_stage = list(model.network.children())[-1]
        return list(last_stage)[-1] if hasattr(last_stage, "__iter__") else last_stage
    if hasattr(model, "blocks"):
        return model.blocks[-1]
    last = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

def grad_cam(model, img_t, class_idx, name):
    model.eval().to(device); img_t = img_t.to(device); img_t.requires_grad_(True)
    acts, grads = [], []
    layer = get_target_layer(model, name)
    def fhook(mod, inp, out):
        acts.append(out); out.register_hook(lambda g: grads.append(g))
    h = layer.register_forward_hook(fhook)
    try:
        logits = model(img_t)
        if class_idx is None:
            class_idx = int(logits.argmax(1).item())
        model.zero_grad(set_to_none=True)
        logits[:, class_idx].sum().backward(retain_graph=True)
        if not acts or not grads:
            return np.zeros((IMG_SIZE, IMG_SIZE))
        a, g = acts[0], grads[0]
        if a.dim() == 4:
            a, g = a[0], g[0]
        if g.dim() < 3:
            return np.zeros((IMG_SIZE, IMG_SIZE))
        cam = F.relu((g.mean(dim=(1, 2), keepdim=True) * a).sum(0))
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                            mode="bilinear", align_corners=False)
        return cam.squeeze().detach().cpu().numpy()
    finally:
        h.remove()

def denorm(t):
    t = t.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    return np.clip(t * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)

def save_gradcam(cnn_m, pool_m, img_t, pred_idx, cls, out):
    cam_c = grad_cam(cnn_m, img_t, pred_idx, "RepVGG")
    cam_p = grad_cam(pool_m, img_t, pred_idx, "PoolFormer")
    img = denorm(img_t)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].imshow(img); ax[0].set_title("Original Image", fontweight="bold")
    ax[1].imshow(img); ax[1].imshow(cam_c, cmap="jet", alpha=0.45)
    ax[1].set_title("RepVGG-A0\n(Local Texture Focus)", fontweight="bold")
    ax[2].imshow(img); ax[2].imshow(cam_p, cmap="jet", alpha=0.45)
    ax[2].set_title("PoolFormer-S12\n(Global Context Focus)", fontweight="bold")
    for a in ax:
        a.axis("off")
    plt.suptitle(f"Grad-CAM - Predicted: {cls}", fontweight="bold", y=1.02)
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches="tight"); plt.close()
    print(f"  Saved: {out}")

def report_efficiency(model, name):
    model.eval().to(device)
    inp = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    if THOP_OK:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            macs, params = profile(model, inputs=(inp,), verbose=False)
        macs_f, params_f = clever_format([macs, params], "%.3f")
    else:
        params_f = f"{sum(p.numel() for p in model.parameters())/1e6:.3f}M"
        macs_f = "N/A"
    with torch.no_grad():
        for _ in range(10):
            model(inp)
    ts = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter(); model(inp)
            if device == "cuda":
                torch.cuda.synchronize()
            ts.append((time.perf_counter() - t0) * 1000)
    mem = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0.0
    print(f"  {name}: params={params_f} | GFLOPs={macs_f} | "
          f"{np.mean(ts):.2f} +/- {np.std(ts):.2f} ms | mem={mem:.2f} MB")
    return float(np.mean(ts)), mem

# =============================================================================
# STEP 1 — TRAIN BACKBONES
# =============================================================================
print(f"\n{'='*60}\n  STEP 1 — FINE-TUNING RepVGG-A0\n{'='*60}")
cnn = train_backbone("repvgg_a0", EPOCHS_CNN, "RepVGG_A0")

print(f"\n{'='*60}\n  STEP 1 — FINE-TUNING PoolFormer-S12\n{'='*60}")
pool = train_backbone("poolformer_s12", EPOCHS_VIT, "PoolFormer_S12")

# =============================================================================
# STEP 2 — WEIGHT SEARCH (VAL ONLY)
# =============================================================================
print(f"\n{'='*60}\n  STEP 2 — WEIGHT SEARCH (VAL ONLY)\n{'='*60}")
P_rep_vl, y_vl = predict_softmax(cnn,  val_loader)
P_pool_vl, _   = predict_softmax(pool, val_loader)
alpha, beta = grid_search_weights(P_rep_vl, P_pool_vl, y_vl)
print(f"  Automatically optimized weights => alpha (RepVGG)={alpha:.2f} | beta (PoolFormer)={beta:.2f}")

# =============================================================================
# STEP 3 — XGBOOST ON STACKED PROBS  (TRAIN -> LOCKED TEST)
# =============================================================================
print(f"\n{'='*60}\n  STEP 3 — XGBOOST (TRAIN -> TEST)\n{'='*60}")
P_rep_tr, y_tr = predict_softmax(cnn,  train_loader_na)
P_pool_tr, _   = predict_softmax(pool, train_loader_na)
P_rep_te, y_te = predict_softmax(cnn,  test_loader)
P_pool_te, _   = predict_softmax(pool, test_loader)

X_tr = stacked_features(P_rep_tr, P_pool_tr, alpha, beta)
X_te = stacked_features(P_rep_te, P_pool_te, alpha, beta)
X_tr_s, y_tr_s = smote_balance(X_tr, y_tr.copy())
xgb = train_xgb(X_tr_s, y_tr_s, X_te, y_te)
proba_te = xgb.predict_proba(X_te)
pred_te  = proba_te.argmax(1)

res = evaluate(y_te, pred_te, proba_te)
print("\n  ===== FINAL TEST-SET RESULTS (HyFuseDP) =====")
for k, v in res.items():
    print(f"     {k:<12}: {v*100:.2f}" if k != "roc_mac" else f"     {k:<12}: {v:.4f}")
print("\n  Classification report (TEST):")
print(classification_report(y_te, pred_te, target_names=class_names))
print("  Confusion matrix (TEST):")
print(confusion_matrix(y_te, pred_te))

# confusion matrix figure
import seaborn as sns
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_te, pred_te), annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("HyFuseDP Confusion Matrix (Test Set)")
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hybrid_confusion_matrix.png"), dpi=200); plt.close()

# per-class ROC
y_bin = label_binarize(y_te, classes=np.arange(NUM_CLASSES))
plt.figure(figsize=(8, 6))
for c in range(NUM_CLASSES):
    try:
        fpr, tpr, _ = roc_curve(y_bin[:, c], proba_te[:, c])
        plt.plot(fpr, tpr, lw=2, label=f"{class_names[c]} (AUC={auc(fpr, tpr):.3f})")
    except Exception:
        pass
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend(loc="lower right", fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hybrid_roc_all_classes.png"), dpi=200); plt.close()

# =============================================================================
# STEP 4 — BASELINES + SOFT-VOTE-ONLY ABLATION (TEST)
# =============================================================================
print(f"\n{'='*60}\n  STEP 4 — BASELINES & ABLATION (TEST)\n{'='*60}")
for bname, P_te in [("RepVGG-A0", P_rep_te), ("PoolFormer-S12", P_pool_te)]:
    r = evaluate(y_te, P_te.argmax(1), P_te)
    print(f"  [{bname}] acc={r['acc']*100:.2f} f1={r['f1_macro']*100:.2f} "
          f"prec={r['precision']*100:.2f} rec={r['recall']*100:.2f} "
          f"kappa={r['kappa']:.4f} mcc={r['mcc']:.4f}")

sv_proba = alpha * P_rep_te + beta * P_pool_te
sv = evaluate(y_te, sv_proba.argmax(1), sv_proba)
print(f"  [SoftVote-only (no XGB)] acc={sv['acc']*100:.2f} f1={sv['f1_macro']*100:.2f} "
      f"prec={sv['precision']*100:.2f} rec={sv['recall']*100:.2f}")
print(f"  [HyFuseDP (full)]        acc={res['acc']*100:.2f} f1={res['f1_macro']*100:.2f} "
      f"prec={res['precision']*100:.2f} rec={res['recall']*100:.2f}")

# =============================================================================
# STEP 5 — McNEMAR (TEST)
# =============================================================================
print(f"\n{'='*60}\n  STEP 5 — McNEMAR (TEST)\n{'='*60}")
from scipy.stats import chi2 as _chi2
def mcnemar(yt, pa, pb, na, nb):
    ca = (np.array(pa) == np.array(yt)); cb = (np.array(pb) == np.array(yt))
    b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb))
    if b + c == 0:
        stat, p = 0.0, 1.0
    else:
        stat = (abs(b - c) - 1) ** 2 / (b + c); p = float(1 - _chi2.cdf(stat, 1))
    sig = "significant (p<0.05)" if p < 0.05 else "not significant"
    print(f"  {na:<16} vs {nb:<16}: b={b:3d} c={c:3d} chi2={stat:.4f} p={p:.4f}  {sig}")
mcnemar(y_te, P_rep_te.argmax(1),  pred_te, "RepVGG-A0",      "HyFuseDP")
mcnemar(y_te, P_pool_te.argmax(1), pred_te, "PoolFormer-S12", "HyFuseDP")
mcnemar(y_te, P_rep_te.argmax(1),  P_pool_te.argmax(1), "RepVGG-A0", "PoolFormer-S12")

# =============================================================================
# STEP 6 — EFFICIENCY
# =============================================================================
print(f"\n{'='*60}\n  STEP 6 — EFFICIENCY\n{'='*60}")
c_ms, c_mem = report_efficiency(cnn, "RepVGG-A0")
p_ms, p_mem = report_efficiency(pool, "PoolFormer-S12")
print(f"  Combined: {c_ms + p_ms:.2f} ms | mem {c_mem + p_mem:.2f} MB | "
      f"params {(sum(p.numel() for p in cnn.parameters())+sum(p.numel() for p in pool.parameters()))/1e6:.3f}M")

# =============================================================================
# STEP 7 — GRAD-CAM + SAVE
# =============================================================================
print(f"\n{'='*60}\n  STEP 7 — GRAD-CAM\n{'='*60}")
for idx, path in enumerate(GRADCAM_IMAGE_PATHS):
    if not os.path.exists(path):
        continue
    img_t = eval_tfms(Image.open(path).convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        pr = torch.softmax(cnn(img_t.to(device)), 1).cpu().numpy()[0]
        pp = torch.softmax(pool(img_t.to(device)), 1).cpu().numpy()[0]
    pidx = int((alpha * pr + beta * pp).argmax())
    save_gradcam(cnn, pool, img_t, pidx, class_names[pidx],
                 os.path.join(OUTPUT_DIR, f"gradcam_compare_{idx}.png"))

torch.save(cnn.state_dict(),  os.path.join(OUTPUT_DIR, "repvgg_a0_finetuned.pth"))
torch.save(pool.state_dict(), os.path.join(OUTPUT_DIR, "poolformer_s12_finetuned.pth"))
print("\n  [DONE] Single seed; all metrics on the LOCKED TEST set; no leakage.")